In [ ]:
!pip install -q efficientnet_pytorch==0.7.1 pillow


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'main'
REPO_DIR = '/kaggle/working/repo'

TILE_DIR = '/kaggle/input/datasets/franciscacarneiro/panda-tiles-36x192-png/panda_tiles_36x192_png'
FOLD = 0
BACKBONE = 'efficientnet-b0'
LOSS = 'ordinal'
N_TILES = 36
TILE_SIZE = 192
FEATURE_TAG_SUFFIX = 'poolv2'  # change to rerun with a fresh weight name
EPOCHS = 6
BATCH_SIZE = None
NUM_WORKERS = None
OUTPUT_DIR = '/kaggle/working'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, REPO_DIR], check=True)

PYTHON = sys.executable

import torch

if not torch.cuda.is_available():
    raise RuntimeError('A real GPU is required for the fold-0 tile run.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
CPU_COUNT = os.cpu_count() or 2

if BATCH_SIZE is None:
    BATCH_SIZE = 8 if GPU_TOTAL_GIB >= 28 else 6 if GPU_TOTAL_GIB >= 20 else 4 if GPU_TOTAL_GIB >= 14 else 2
if NUM_WORKERS is None:
    NUM_WORKERS = min(4, max(2, CPU_COUNT // 2))

FEATURE_TAG = f'tiles{N_TILES}_imsize{TILE_SIZE}'
if FEATURE_TAG_SUFFIX:
    FEATURE_TAG = f'{FEATURE_TAG}_{FEATURE_TAG_SUFFIX}'
EXPECTED_WEIGHT = f"{BACKBONE.replace('-', '')}_{FEATURE_TAG}_{LOSS}_fold{FOLD}.pth"

def count_tile_artifacts(path_value):
    try:
        names = os.listdir(path_value)
    except OSError:
        return 0
    return sum(name.lower().endswith(('.npy', '.npz', '.png')) for name in names)

def find_tile_dirs(min_count=1000):
    matches = []
    for root, _, files in os.walk('/kaggle/input'):
        count = sum(name.lower().endswith(('.npy', '.npz', '.png')) for name in files)
        if count >= min_count:
            matches.append((root, count))
    matches.sort(key=lambda item: (0 if 'tile' in item[0].lower() else 1, -item[1], item[0]))
    return matches

if TILE_DIR is None:
    tile_candidates = find_tile_dirs()
    if not tile_candidates:
        raise RuntimeError('No attached tile dataset found under /kaggle/input. Attach the tile dataset first.')
    TILE_DIR = tile_candidates[0][0]
    print('Auto-selected TILE_DIR:', TILE_DIR)
    print('Candidate count:', tile_candidates[0][1])
else:
    if not os.path.isdir(TILE_DIR):
        raise RuntimeError(f'TILE_DIR does not exist: {TILE_DIR}')
    print('Using TILE_DIR:', TILE_DIR)

print('GPU_NAME       =', GPU_NAME)
print('GPU_TOTAL_GIB  =', f'{GPU_TOTAL_GIB:.1f}')
print('CPU_COUNT      =', CPU_COUNT)
print('BATCH_SIZE     =', BATCH_SIZE)
print('NUM_WORKERS    =', NUM_WORKERS)
print('TAG_SUFFIX     =', FEATURE_TAG_SUFFIX)
print('FEATURE_TAG    =', FEATURE_TAG)
print('EXPECTED_WEIGHT=', EXPECTED_WEIGHT)
print('TILE_COUNT     =', count_tile_artifacts(TILE_DIR))


In [ ]:
sample_files = sorted(os.listdir(TILE_DIR))[:10]
print('First 10 tile artifacts:')
for name in sample_files:
    print(' ', name)


In [ ]:
cmd = [
    PYTHON, '-m', 'src.train',
    '--fold', str(FOLD),
    '--folds-csv', os.path.join(REPO_DIR, 'data', 'train_folds.csv'),
    '--tile-dir', TILE_DIR,
    '--backbone', BACKBONE,
    '--loss', LOSS,
    '--n-tiles', str(N_TILES),
    '--tile-size', str(TILE_SIZE),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--feature-tag', FEATURE_TAG,
    '--amp',
    '--pin-memory',
    '--output-dir', OUTPUT_DIR,
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.pth'):
        print(f, os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6, 'MB')


In [ ]:
print('Paste this row into results.md after the run finishes and replace <VAL_QWK>:')
print()
print(f"| 2026-06-06 | B | tiles36-imsize192-b0-ordinal-fold0 | effnet-b0, 36 tiles, 192 tile size, fold 0, 6 epochs, lr 3e-4, ordinal BCE, dropout 0.3 | <VAL_QWK> | — | — | First real concat-tile-pooling run on Track A tile artifacts. Saved `{os.path.join(OUTPUT_DIR, EXPECTED_WEIGHT)}`. |")
